# 04 — Activation Patching

Correlation is not causation — patching is. We cache activations from a
*clean* prompt, splice them into a *corrupted* run, and measure how much
of the clean behaviour returns (logit-difference recovery, as in ROME/IOI).

In [ ]:
import sys
sys.path.insert(0, "..")  # run from the notebooks/ directory

import matplotlib
import torch

import kamui
from kamui.model.config import ModelConfig

torch.manual_seed(0)
print("KAMUI", kamui.__version__)

In [ ]:
# Train a tiny model on a synthetic corpus (~30s on CPU).
# The corpus is a seeded word-salad: repetitive enough to learn, varied enough
# that BPE cannot collapse it into a handful of giant tokens.
import random

from kamui.tokenizer.bpe import BPETokenizer
from kamui.training import DataLoader, TextDataset, Trainer, TrainingConfig

rng = random.Random(0)
WORDS = ["the", "cat", "dog", "sat", "ran", "on", "to", "mat", "log", "sun"]
CORPUS = " ".join(rng.choice(WORDS) for _ in range(4000))

config = ModelConfig(n_layers=2, d_model=64, n_heads=4, d_ff=128,
                     vocab_size=300, context_length=32, dropout=0.0)
tokenizer = BPETokenizer.train(CORPUS, vocab_size=config.vocab_size)
tokens = tokenizer.encode(CORPUS)

model = kamui.KAMUITransformer(config)
trainer = Trainer(
    model,
    DataLoader(TextDataset(tokens, config.context_length), batch_size=8, seed=0),
    config=TrainingConfig(max_lr=3e-3, warmup_steps=10, max_steps=1000),
)
records = trainer.train(150)
model.eval()
print(f"loss: {records[0]['train_loss']:.3f} -> {records[-1]['train_loss']:.3f}")

In [ ]:
from kamui.mechinterp import ActivationPatcher

clean = torch.tensor(tokenizer.encode("the cat ran to the"))
corrupted = torch.tensor(tokenizer.encode("the dog ran to the"))
n = min(len(clean), len(corrupted))
patcher = ActivationPatcher(model)

# Sanity anchor: patching the embedding restores the clean run exactly.
print("embed recovery:", patcher.patch_single(clean[:n], corrupted[:n], "embed.output"))

In [ ]:
# Which layer's attention carries the cat/dog distinction?
result = patcher.patch_all_layers(clean[:n], corrupted[:n], component="attn")
print("best layer:", result.best_layer())
result.plot()

In [ ]:
# Zoom to head resolution.
patcher.patch_all_heads(clean[:n], corrupted[:n]).plot()